# YOLO-CLIP: WILDS-trained image-level adaptation

Task 1 only: prepare the Molab environment, load iWildCam-WILDS v2.0, and check its labels and development splits. The main experiment is training an integrated YOLO-CLIP model from WILDS image labels, initialized from pretrained YOLO-World. No model weights or test splits are used in Task 1.

## Runtime inventory

Run the numbered cells in order in Molab. Report outputs by `CELL` number. Package *installed* versions can differ from versions already loaded in the current Python process; both are recorded where relevant.

In [ ]:
# CELL 1
import platform
import sys
from importlib import metadata
from pathlib import Path

print('Python:', sys.version.splitlines()[0])
print('Platform:', platform.platform())
print('Working directory:', Path.cwd())
for package_name in ('torch', 'torchvision', 'ultralytics', 'clip', 'wilds',
                     'huggingface-hub', 'numpy', 'pandas', 'Pillow'):
    try:
        installed_version = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        installed_version = 'not installed'
    print(f'{package_name}: {installed_version}')

In [ ]:
# CELL 2
import shutil

gib = 1024 ** 3
disk = shutil.disk_usage(Path.cwd())
print(f'Working-filesystem space: {disk.free / gib:.1f} GiB free / {disk.total / gib:.1f} GiB total')
print('This is a runtime snapshot, not a guarantee of persistent storage.')
try:
    import torch
except (ImportError, OSError) as torch_exc:
    print('PyTorch could not be imported:', repr(torch_exc))
else:
    print('Loaded PyTorch:', torch.__version__)
    print('PyTorch CUDA build:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        for device_index in range(torch.cuda.device_count()):
            properties = torch.cuda.get_device_properties(device_index)
            print(f'GPU {device_index}: {properties.name}; VRAM: {properties.total_memory / gib:.1f} GiB')

## WILDS data location

The previous Molab run used `/marimo/data/iwildcam_v2.0`. Change `DATA_ROOT` only if this session stores the dataset elsewhere. CELL 3 does not download anything. CELL 5 reuses a verified extraction when present; otherwise it downloads the pinned Hugging Face mirror archive and verifies its size, SHA-256, metadata hash, and official split counts before loading WILDS.

In [ ]:
# CELL 3
DATA_ROOT = (Path.cwd() / 'data').resolve()  # Change only if this session stores WILDS elsewhere.
DATASET_DIR = DATA_ROOT / 'iwildcam_v2.0'
print('WILDS root directory:', DATA_ROOT)
print('Expected iWildCam directory:', DATASET_DIR)
print('Dataset directory exists:', DATASET_DIR.is_dir())
print('metadata.csv exists:', (DATASET_DIR / 'metadata.csv').is_file())
print('categories.csv exists:', (DATASET_DIR / 'categories.csv').is_file())
if not (DATASET_DIR / 'metadata.csv').is_file():
    print('Dataset is not ready at this path; no download was attempted yet.')

In [ ]:
# CELL 4
import subprocess

try:
    wilds_installed = metadata.version('wilds')
except metadata.PackageNotFoundError:
    wilds_installed = None
if wilds_installed != '2.0.0':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'wilds==2.0.0'], check=True)
try:
    hub_installed = metadata.version('huggingface-hub')
except metadata.PackageNotFoundError:
    hub_installed = None
if hub_installed is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'huggingface_hub'], check=True)
import wilds
print('Loaded WILDS:', wilds.__version__)
print('huggingface_hub installed:', metadata.version('huggingface-hub'))

In [ ]:
# CELL 5
import hashlib
import tarfile

import pandas as pd
from huggingface_hub import hf_hub_download
from wilds import get_dataset

MIRROR_REPO = 'emmilly/iwildcam-wilds-v2'
MIRROR_REVISION = '1c01b321d444aa8a15e0bc7a0204d8c6559c8c5b'
ARCHIVE_SHA256 = '05c872f46eee44315f79bc90f4098c69ae3799a2b95116db02292ea1b4e370d2'
ARCHIVE_BYTES = 11_957_415_473
RELEASE_MARKER = DATASET_DIR / 'RELEASE_v2.0.txt'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
required_files = (DATASET_DIR / 'metadata.csv', DATASET_DIR / 'categories.csv', RELEASE_MARKER)
if not (all(path.is_file() for path in required_files) and (DATASET_DIR / 'train').is_dir()):
    print('Downloading the pinned iWildCam v2.0 archive from', MIRROR_REPO)
    archive = Path(hf_hub_download(
        repo_id=MIRROR_REPO, filename='iwildcam_v2.0.tar.gz', repo_type='dataset',
        revision=MIRROR_REVISION, local_dir=str(DATA_ROOT),
    ))
    assert archive.stat().st_size == ARCHIVE_BYTES, 'Archive size differs from mirror manifest'
    digest = hashlib.sha256()
    with archive.open('rb') as source:
        for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    assert digest.hexdigest() == ARCHIVE_SHA256, 'Archive SHA-256 differs from mirror manifest'
    print('Archive size and SHA-256 verified. Extracting into', DATASET_DIR)
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as tar:
        tar.extractall(DATASET_DIR, filter='data')

metadata_file = DATASET_DIR / 'metadata.csv'
assert metadata_file.is_file() and (DATASET_DIR / 'categories.csv').is_file()
assert (DATASET_DIR / 'train').is_dir(), 'Image directory is missing'
metadata_md5 = hashlib.md5(metadata_file.read_bytes()).hexdigest()
assert metadata_md5 == 'ab4905ee40ac45ab66ddb176d473a4b4', 'Metadata differs from mirror manifest'
split_counts = pd.read_csv(metadata_file, usecols=['split'])['split'].value_counts().to_dict()
expected_counts = {'train': 129809, 'val': 14961, 'test': 42791,
                   'id_val': 7314, 'id_test': 8154}
assert split_counts == expected_counts, f'Unexpected official split sizes: {split_counts}'
if not RELEASE_MARKER.is_file():
    RELEASE_MARKER.touch()  # WILDS recognizes a verified manual extraction.
dataset = get_dataset(dataset='iwildcam', version='2.0',
                      root_dir=str(DATA_ROOT), download=False)
print('Dataset ready:', DATASET_DIR, '| classes:', dataset.n_classes)

## Official split and class mapping

List all five official split names/counts, but load subsets only for `train`, `id_val`, and `val`. Do not read `id_test` or `test` images or labels. `categories.csv` includes extra catalog rows; WILDS labels are its `y` values from 0 to 181, not its separate `category_id` column.

In [ ]:
# CELL 6
DEVELOPMENT_SPLITS = ('train', 'id_val', 'val')
print('Number of classes:', dataset.n_classes)
print('Metadata fields:', dataset.metadata_fields)
assert set(expected_counts) == set(dataset.split_dict), 'Unexpected official split names'
for split_name in ('train', 'id_val', 'val', 'id_test', 'test'):
    print(f'{split_name:7s} | {dataset.split_names[split_name]:22s} | '
          f'{expected_counts[split_name]:,} images')
for development_split in DEVELOPMENT_SPLITS:
    print(f'Loaded {development_split}: {len(dataset.get_subset(development_split)):,} images')
print('Final test subsets were not loaded.')

In [ ]:
# CELL 7
categories = pd.read_csv(DATASET_DIR / 'categories.csv')
print('Category columns:', list(categories.columns))
print(categories.head(5).to_string(index=True))
assert {'y', 'name'} <= set(categories.columns), 'Inspect categories.csv columns'
active_categories = categories[categories['y'].between(0, dataset.n_classes - 1)].copy()
print('Catalog rows outside WILDS class range:', len(categories) - len(active_categories))
assert active_categories['y'].is_unique, 'Duplicate active WILDS class IDs'
id_to_name = dict(zip(active_categories['y'].astype(int),
                      active_categories['name'].astype(str)))
assert set(id_to_name) == set(range(dataset.n_classes)), 'WILDS class map is incomplete'
assert id_to_name[0].strip().lower() == 'empty', 'Expected empty label at y=0'
print('Mapped classes:', len(id_to_name), '| empty class ID: 0')
print('First 12 classes:', [(i, id_to_name[i]) for i in range(12)])

## Task 2: inspect the integrated model's training interface

CELLS 8-10 install the pinned Ultralytics release, load and verify the YOLO-World S-v2 starting checkpoint, then inspect the model's text-feature and pre-NMS score interfaces. These cells do **not** train the model, encode WILDS class names, or evaluate images. We will add the embedding-gradient smoke test after reviewing their Molab output.

In [ ]:
# CELL 8
ULTRALYTICS_VERSION = '8.4.178'
try:
    installed_ultralytics = metadata.version('ultralytics')
except metadata.PackageNotFoundError:
    installed_ultralytics = None
if installed_ultralytics != ULTRALYTICS_VERSION:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', f'ultralytics=={ULTRALYTICS_VERSION}'],
        check=True,
    )
import ultralytics
print('Ultralytics:', ultralytics.__version__)
print('Loaded PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert ultralytics.__version__ == ULTRALYTICS_VERSION
assert torch.cuda.is_available(), 'Task 2 expects the Molab GPU'

In [ ]:
# CELL 9
from ultralytics import YOLOWorld

STARTING_WEIGHTS = 'yolov8s-worldv2.pt'
STARTING_SHA256 = '9b2c17ab6124a913e9b3a5c170617920d91b0f01111a8479da69f00e2cf27792'
TRAIN_IMAGE_SIZE = 640
training_world = YOLOWorld(STARTING_WEIGHTS)  # Downloads the starting weights if absent.
training_world.model.eval()
training_world.model.requires_grad_(False)
starting_checkpoint = Path(STARTING_WEIGHTS).resolve()
assert starting_checkpoint.is_file(), f'Checkpoint not found: {starting_checkpoint}'
starting_digest = hashlib.sha256()
with starting_checkpoint.open('rb') as checkpoint_file:
    for checkpoint_chunk in iter(lambda: checkpoint_file.read(8 * 1024 * 1024), b''):
        starting_digest.update(checkpoint_chunk)
assert starting_digest.hexdigest() == STARTING_SHA256, 'Starting checkpoint SHA-256 differs'
print('Starting checkpoint:', starting_checkpoint)
print('Bytes:', starting_checkpoint.stat().st_size)
print('SHA-256 verified:', starting_digest.hexdigest())
print('Planned input size:', TRAIN_IMAGE_SIZE)
print('Pretrained parameters frozen:',
      sum(p.numel() for p in training_world.model.parameters() if p.requires_grad) == 0)

In [ ]:
# CELL 10
import inspect

from ultralytics.nn.modules.head import WorldDetect

integrated_model = training_world.model
detection_head = integrated_model.model[-1]
cached_text_features = getattr(integrated_model, 'txt_feats', None)
print('Integrated model:', type(integrated_model).__name__)
print('Detection head:', type(detection_head).__name__)
print('WorldDetect head:', isinstance(detection_head, WorldDetect))
print('Model predict signature:', inspect.signature(integrated_model.predict))
print('Head forward signature:', inspect.signature(detection_head.forward))
print('Cached text-feature shape:',
      None if cached_text_features is None else tuple(cached_text_features.shape))
print('Cached text features are an nn.Parameter:',
      isinstance(cached_text_features, torch.nn.Parameter))
print('CLIP text encoder currently loaded:',
      getattr(integrated_model, 'clip_model', None) is not None)
head_forward_lines = inspect.getsource(type(detection_head).forward).splitlines()
print('Head forward implementation (first 55 lines):')
print('\n'.join(head_forward_lines[:55]))
assert isinstance(detection_head, WorldDetect), 'Expected integrated WorldDetect head'
assert 'txt_feats' in inspect.signature(integrated_model.predict).parameters, (
    'The pinned model does not expose direct text-feature input'
)

### Initialize WILDS class embeddings

CELL 11 installs the pinned Ultralytics CLIP fork **without pulling a second PyTorch build**. CELL 12 encodes the 181 supplied non-`empty` WILDS class names once, then freezes the encoder and detector. We train the resulting vectors later; the class-name words themselves are not learned. The first encoding may download the CLIP ViT-B/32 text-encoder weights.

In [ ]:
# CELL 11
try:
    ftfy_installed = metadata.version('ftfy')
except metadata.PackageNotFoundError:
    ftfy_installed = None
if ftfy_installed is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'ftfy>=6.3,<7'], check=True)
try:
    regex_installed = metadata.version('regex')
except metadata.PackageNotFoundError:
    regex_installed = None
if regex_installed is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'regex'], check=True)
try:
    clip_installed = metadata.version('clip')
except metadata.PackageNotFoundError:
    clip_installed = None
if clip_installed is None:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-deps',
        'git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40',
    ], check=True)
import clip
assert hasattr(clip, 'load') and hasattr(clip, 'tokenize'), 'Unexpected CLIP package'
print('CLIP package:', metadata.version('clip'))
print('CLIP module:', clip.__file__)
print('Loaded PyTorch remains:', torch.__version__)

In [ ]:
# CELL 12
animal_class_ids = tuple(range(1, dataset.n_classes))
animal_prompts = tuple(id_to_name[y].strip() for y in animal_class_ids)
assert len(animal_prompts) == 181 and len(set(animal_prompts)) == 181
assert all(animal_prompts), 'Blank WILDS class name'
training_world.set_classes(list(animal_prompts))
integrated_model.eval()
integrated_model.requires_grad_(False)
wilds_text_features = integrated_model.txt_feats
assert tuple(wilds_text_features.shape[:2]) == (1, len(animal_prompts))
assert detection_head.nc == len(animal_prompts)
assert all(training_world.names[i] == animal_prompts[i]
           for i in range(len(animal_prompts)))
assert all(not p.requires_grad for p in integrated_model.parameters())
print('WILDS animal classes:', len(animal_prompts), '| empty image label: 0')
print('Class 0 -> WILDS y=1:', animal_prompts[0])
print('Class 180 -> WILDS y=181:', animal_prompts[-1])
print('Text-feature shape:', tuple(wilds_text_features.shape))
print('CLIP encoder loaded:', getattr(integrated_model, 'clip_model', None) is not None)
print('Pretrained trainable parameters:',
      sum(p.numel() for p in integrated_model.parameters() if p.requires_grad))

### Pre-NMS scores and gradient smoke test

Use **one training image**, not a validation or test image. CELL 13 applies YOLO-style letterboxing and inspects the raw `scores` tensor. CELL 14 treats the initialized class vectors as a temporary trainable parameter, backpropagates one class score, and checks that a changed vector changes its score. It performs **no optimizer step**, does not update detector weights, and does not use a box label.

In [ ]:
# CELL 13
import numpy as np
from ultralytics.data.augment import LetterBox

probe_subset = dataset.get_subset('train')
probe_index = int(torch.nonzero(probe_subset.y_array != 0, as_tuple=False)[0].item())
probe_image, probe_y, probe_metadata = probe_subset[probe_index]
probe_y = int(probe_y)
assert 1 <= probe_y < dataset.n_classes
letterbox = LetterBox(new_shape=(TRAIN_IMAGE_SIZE, TRAIN_IMAGE_SIZE), auto=False)
probe_pixels = letterbox(image=np.asarray(probe_image.convert('RGB')))
probe_tensor = (torch.from_numpy(probe_pixels.copy()).permute(2, 0, 1)
                .contiguous().unsqueeze(0).float().div(255).to('cuda:0'))
integrated_model.to(probe_tensor.device).eval()
assert next(integrated_model.parameters()).device == probe_tensor.device
with torch.no_grad():
    probe_output = integrated_model.predict(probe_tensor)
assert isinstance(probe_output, tuple) and len(probe_output) == 2
probe_raw = probe_output[1]
assert isinstance(probe_raw, dict) and 'scores' in probe_raw
probe_scores = probe_raw['scores']
assert probe_scores.ndim == 3 and probe_scores.shape[:2] == (1, len(animal_prompts))
print('Training image index:', probe_index, '| WILDS y:', probe_y, id_to_name[probe_y])
print('Original image size:', probe_image.size, '| model input:', tuple(probe_tensor.shape))
print('Raw pre-NMS class-score shape:', tuple(probe_scores.shape))
print('Candidate positions per class:', probe_scores.shape[-1])
print('Head output keys:', sorted(probe_raw))
print('Scores finite:', bool(torch.isfinite(probe_scores).all()))

In [ ]:
# CELL 14
trial_embeddings = torch.nn.Parameter(
    wilds_text_features.detach().to(device=probe_tensor.device, dtype=probe_tensor.dtype).clone()
)
trial_output = integrated_model.predict(probe_tensor, txt_feats=trial_embeddings)
trial_scores = trial_output[1]['scores']
assert trial_scores.shape == probe_scores.shape and trial_scores.requires_grad
trial_class_index = probe_y - 1  # Detector index 0 corresponds to WILDS y=1.
trial_image_score = trial_scores[0, trial_class_index].amax()
trial_image_score.backward()
trial_grad_norm = float(trial_embeddings.grad[0, trial_class_index].norm().item())
frozen_grad_names = [name for name, parameter in integrated_model.named_parameters()
                     if parameter.grad is not None]
with torch.no_grad():
    perturbed_embeddings = trial_embeddings.detach().clone()
    perturbed_embeddings[0, trial_class_index] = torch.roll(
        perturbed_embeddings[0, trial_class_index], shifts=1, dims=0
    )
    perturbed_output = integrated_model.predict(probe_tensor, txt_feats=perturbed_embeddings)
    perturbed_score = perturbed_output[1]['scores'][0, trial_class_index].amax()
score_change = abs(float((perturbed_score - trial_image_score.detach()).item()))
print('Trial WILDS class:', probe_y, id_to_name[probe_y])
print('Top raw region score:', float(trial_image_score.detach().item()))
print('Class-embedding gradient norm:', trial_grad_norm)
print('Score change after altering that embedding:', score_change)
print('Frozen model parameters with gradients:', frozen_grad_names)
assert trial_grad_norm > 0 and np.isfinite(trial_grad_norm), 'No usable embedding gradient'
assert score_change > 1e-6, 'Changing the embedding did not alter the image score'
assert not frozen_grad_names, 'A frozen model parameter received a gradient'
print('Differentiable embedding path verified; no optimizer step or weight update.')

## Task 3: image-level training objective and first recipe

The model produces one raw score per non-`empty` class and candidate position. For each such class, take the maximum over positions; place a **fixed zero logit** at index 0 for WILDS `empty`; then use ordinary cross-entropy against the WILDS image label. This is weak image-level supervision: it does not know which box contains the subject, and an image may contain more than one animal even though WILDS supplies one target label. No NMS or box annotation enters this loss.

First-run recipe to review before a long job: initialize the 181 class vectors from the existing WILDS-name text embeddings; freeze all pretrained weights; AdamW on the vectors only, constant learning rate `1e-4`, weight decay `0`, batch size `16`, no gradient accumulation, gradient clipping at norm `1`, up to `5` epochs, seed `785`, checkpoint each epoch. Use 640-pixel letterboxing without additional augmentation, natural training frequencies, and unweighted cross-entropy first. Select the checkpoint by lowest `id_val` image-level loss. For the final image prediction, use the highest-scoring post-NMS detection and a separately selected `val` threshold for `empty`; report `val` results as threshold-tuned. If Task 4 shows an out-of-memory or throughput problem, record and revise the batch/epoch choice before full training.

In [ ]:
# CELL 15
TRAIN_RECIPE = {
    'seed': 785,
    'image_size': TRAIN_IMAGE_SIZE,
    'trainable': '181 CLIP-initialized WILDS class vectors only',
    'region_aggregation': 'maximum pre-NMS raw score per class',
    'empty_logit': 0.0,
    'loss': 'unweighted image-level cross-entropy',
    'optimizer': 'AdamW',
    'learning_rate': 1e-4,
    'learning_rate_schedule': 'constant',
    'weight_decay': 0.0,
    'preprocessing': '640-pixel letterbox; no additional augmentation',
    'batch_size': 16,
    'gradient_accumulation': 1,
    'gradient_clip_norm': 1.0,
    'max_epochs': 5,
    'checkpoint_every_epochs': 1,
    'checkpoint_selection': 'lowest id_val image-level cross-entropy',
    'threshold_selection': 'val macro-F1; report as tuned',
    'final_image_rule': 'highest post-NMS detection or empty below threshold',
}
torch.manual_seed(TRAIN_RECIPE['seed'])
assert TRAIN_RECIPE['image_size'] == 640
assert len(animal_class_ids) == 181 and animal_class_ids == tuple(range(1, 182))
assert all(not p.requires_grad for p in integrated_model.parameters())
for recipe_key, recipe_value in TRAIN_RECIPE.items():
    print(f'{recipe_key}: {recipe_value}')

In [ ]:
# CELL 16
import torch.nn.functional as F

def image_level_logits(raw_class_scores):
    # raw_class_scores: [batch, 181 animal classes, candidate positions]
    assert raw_class_scores.ndim == 3 and raw_class_scores.shape[1] == 181
    animal_logits = raw_class_scores.amax(dim=-1)
    empty_logits = torch.zeros_like(animal_logits[:, :1])
    return torch.cat((empty_logits, animal_logits), dim=1)

synthetic_scores = torch.full((2, 181, 3), -2.0)
synthetic_scores[0, 23, 1] = 4.0  # Detector 23 -> WILDS y=24.
synthetic_logits = image_level_logits(synthetic_scores)
assert synthetic_logits.shape == (2, 182)
assert synthetic_logits[0].argmax().item() == 24
assert synthetic_logits[1].argmax().item() == 0
synthetic_targets = torch.tensor([24, 0], dtype=torch.long)
synthetic_loss = F.cross_entropy(synthetic_logits, synthetic_targets)
assert torch.isfinite(synthetic_loss)
print('Image-logit shape:', tuple(synthetic_logits.shape))
print('Synthetic predictions:', synthetic_logits.argmax(dim=1).tolist())
print('Synthetic animal/empty cross-entropy:', float(synthetic_loss.item()))

### Two-image loss smoke test

CELL 17 combines the already-inspected animal training image with one `empty` training image. It computes the proposed loss and checks that both labels fit the 182-logit mapping. Backward is only a feasibility check; no optimizer or model update occurs. Initial predictions are diagnostics, not an accuracy estimate.

In [ ]:
# CELL 17
empty_probe_index = int(torch.nonzero(probe_subset.y_array == 0, as_tuple=False)[0].item())
empty_probe_image, empty_probe_y, empty_probe_metadata = probe_subset[empty_probe_index]
assert int(empty_probe_y) == 0
empty_probe_pixels = letterbox(image=np.asarray(empty_probe_image.convert('RGB')))
empty_probe_tensor = (torch.from_numpy(empty_probe_pixels.copy()).permute(2, 0, 1)
                      .contiguous().unsqueeze(0).float().div(255).to(probe_tensor.device))
loss_images = torch.cat((probe_tensor, empty_probe_tensor), dim=0)
loss_targets = torch.tensor([probe_y, 0], dtype=torch.long, device=probe_tensor.device)
loss_embeddings = torch.nn.Parameter(
    wilds_text_features.detach().to(device=probe_tensor.device, dtype=probe_tensor.dtype).clone()
)
loss_output = integrated_model.predict(loss_images, txt_feats=loss_embeddings)
loss_raw_scores = loss_output[1]['scores']
loss_logits = image_level_logits(loss_raw_scores)
assert loss_logits.shape == (2, dataset.n_classes)
loss_value = F.cross_entropy(loss_logits, loss_targets)
assert torch.isfinite(loss_value), 'Image-level loss is not finite'
loss_value.backward()
loss_embedding_grad_norm = float(loss_embeddings.grad.norm().item())
loss_frozen_grad_names = [name for name, parameter in integrated_model.named_parameters()
                          if parameter.grad is not None]
print('Animal/empty train indices:', probe_index, empty_probe_index)
print('True WILDS labels:', loss_targets.tolist())
print('Initial image-logit predictions:', loss_logits.argmax(dim=1).tolist())
print('Logit tensor:', tuple(loss_logits.shape), '| loss:', float(loss_value.item()))
print('Class-vector gradient norm:', loss_embedding_grad_norm)
print('Frozen detector parameters with gradients:', loss_frozen_grad_names)
assert np.isfinite(loss_embedding_grad_norm) and loss_embedding_grad_norm > 0
assert not loss_frozen_grad_names
print('Image-level objective is differentiable; no optimizer step or weights update.')

## Task 4: training-pipeline smoke test

CELLS 18-19 build a deterministic image-label loader and make **one** optimizer step on a training batch. This checks actual device memory, step time, and parameter isolation before we attempt a tiny-subset overfit or a full run. The official `id_val`, `val`, and test images are not used here.

In [ ]:
# CELL 18
from torch.utils.data import DataLoader, Dataset, Subset

class WILDSImageLabels(Dataset):
    def __init__(self, subset, image_size):
        self.subset = subset
        self.letterbox = LetterBox(new_shape=(image_size, image_size), auto=False)

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label, image_metadata = self.subset[index]
        pixels = self.letterbox(image=np.asarray(image.convert('RGB')))
        image_tensor = (torch.from_numpy(pixels.copy()).permute(2, 0, 1)
                        .contiguous().float().div(255))
        return image_tensor, int(label), torch.as_tensor(image_metadata).long()

train_image_data = WILDSImageLabels(probe_subset, TRAIN_RECIPE['image_size'])
loader_generator = torch.Generator().manual_seed(TRAIN_RECIPE['seed'])
train_loader = DataLoader(
    train_image_data, batch_size=TRAIN_RECIPE['batch_size'], shuffle=True,
    num_workers=0, pin_memory=True, generator=loader_generator,
)
tiny_train_data = Subset(train_image_data, [empty_probe_index, probe_index])
tiny_train_loader = DataLoader(tiny_train_data, batch_size=2, shuffle=False, num_workers=0)
tiny_images_cpu, tiny_labels_cpu, tiny_metadata_cpu = next(iter(tiny_train_loader))
train_images_cpu, train_labels_cpu, train_metadata_cpu = next(iter(train_loader))
assert len(train_image_data) == 129809 and len(train_loader) > 0
assert tiny_labels_cpu.tolist() == [0, probe_y]
assert train_images_cpu.shape == (TRAIN_RECIPE['batch_size'], 3, 640, 640)
assert train_labels_cpu.min() >= 0 and train_labels_cpu.max() < dataset.n_classes
assert train_metadata_cpu.shape[0] == TRAIN_RECIPE['batch_size']
print('Training examples:', len(train_image_data), '| batches per epoch:', len(train_loader))
print('Full batch image tensor:', tuple(train_images_cpu.shape),
      '| labels:', train_labels_cpu.tolist())
print('Tiny animal/empty labels:', tiny_labels_cpu.tolist())
print('Metadata fields:', dataset.metadata_fields)
print('Tiny camera locations:', tiny_metadata_cpu[:, 0].tolist())
print('No box targets, validation images, or test images were loaded.')

In [ ]:
# CELL 19
import time

integrated_model.eval()  # Keep frozen batch-normalization statistics fixed.
assert all(not p.requires_grad for p in integrated_model.parameters())
train_vectors = torch.nn.Parameter(
    wilds_text_features.detach().to(device=probe_tensor.device, dtype=torch.float32).clone()
)
smoke_optimizer = torch.optim.AdamW(
    [train_vectors], lr=TRAIN_RECIPE['learning_rate'],
    weight_decay=TRAIN_RECIPE['weight_decay'],
)
frozen_parameter_snapshot = {
    name: parameter.detach().clone() for name, parameter in integrated_model.named_parameters()
}
vectors_before_step = train_vectors.detach().clone()
train_images_gpu = train_images_cpu.to(probe_tensor.device, non_blocking=True)
train_labels_gpu = train_labels_cpu.to(probe_tensor.device, non_blocking=True)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
step_start = time.perf_counter()
smoke_optimizer.zero_grad(set_to_none=True)
step_output = integrated_model.predict(train_images_gpu, txt_feats=train_vectors)
step_logits = image_level_logits(step_output[1]['scores'])
step_loss = F.cross_entropy(step_logits, train_labels_gpu)
assert torch.isfinite(step_loss), 'Training loss is not finite'
step_loss.backward()
gradient_norm_before_clip = torch.nn.utils.clip_grad_norm_(
    [train_vectors], TRAIN_RECIPE['gradient_clip_norm']
)
assert torch.isfinite(gradient_norm_before_clip)
smoke_optimizer.step()
torch.cuda.synchronize()
step_seconds = time.perf_counter() - step_start
step_peak_gib = torch.cuda.max_memory_allocated() / gib
vector_max_change = float((train_vectors.detach() - vectors_before_step).abs().max().item())
changed_frozen_parameters = [
    name for name, parameter in integrated_model.named_parameters()
    if not torch.equal(parameter.detach(), frozen_parameter_snapshot[name])
]
assert vector_max_change > 0, 'Optimizer did not update the class vectors'
assert not changed_frozen_parameters, 'A frozen detector parameter changed'
print('Batch size:', len(train_labels_gpu), '| image logits:', tuple(step_logits.shape))
print('Loss before update:', float(step_loss.detach().item()))
print('Gradient norm before clipping:', float(gradient_norm_before_clip.item()))
print('Maximum class-vector change:', vector_max_change)
print('Changed frozen detector parameters:', changed_frozen_parameters)
print(f'One optimizer step: {step_seconds:.2f} s | peak allocated GPU memory: {step_peak_gib:.2f} GiB')
print('This is a training smoke test, not the full training run.')

**Tiny-set overfit check.** Reinitialize the class vectors and repeatedly train on the same empty/cow pair. This checks whether the image-level objective can learn both labels before we commit to the full training run.

In [ ]:
# CELL 20
tiny_vectors = torch.nn.Parameter(
    wilds_text_features.detach().to(device=probe_tensor.device, dtype=torch.float32).clone()
)
tiny_optimizer = torch.optim.AdamW(
    [tiny_vectors], lr=TRAIN_RECIPE['learning_rate'],
    weight_decay=TRAIN_RECIPE['weight_decay'],
)
tiny_images_gpu = tiny_images_cpu.to(probe_tensor.device, non_blocking=True)
tiny_labels_gpu = tiny_labels_cpu.to(probe_tensor.device, non_blocking=True)
assert tiny_labels_gpu.tolist() == [0, probe_y]
assert all(not p.requires_grad for p in integrated_model.parameters())
integrated_model.eval()
with torch.no_grad():
    initial_output = integrated_model.predict(tiny_images_gpu, txt_feats=tiny_vectors)
    initial_logits = image_level_logits(initial_output[1]['scores'])
    initial_loss = float(F.cross_entropy(initial_logits, tiny_labels_gpu).item())
    initial_predictions = initial_logits.argmax(dim=1).tolist()
print('Tiny labels:', tiny_labels_gpu.tolist(), '| initial predictions:', initial_predictions)
print(f'Initial tiny-set loss: {initial_loss:.4f}')
tiny_start = time.perf_counter()
tiny_history = []
for tiny_step in range(1, 201):
    tiny_optimizer.zero_grad(set_to_none=True)
    tiny_output = integrated_model.predict(tiny_images_gpu, txt_feats=tiny_vectors)
    tiny_logits = image_level_logits(tiny_output[1]['scores'])
    tiny_loss = F.cross_entropy(tiny_logits, tiny_labels_gpu)
    assert torch.isfinite(tiny_loss), f'Non-finite tiny-set loss at step {tiny_step}'
    tiny_loss.backward()
    torch.nn.utils.clip_grad_norm_([tiny_vectors], TRAIN_RECIPE['gradient_clip_norm'])
    tiny_optimizer.step()
    if tiny_step == 1 or tiny_step % 25 == 0:
        with torch.no_grad():
            check_output = integrated_model.predict(tiny_images_gpu, txt_feats=tiny_vectors)
            check_logits = image_level_logits(check_output[1]['scores'])
            check_loss = float(F.cross_entropy(check_logits, tiny_labels_gpu).item())
            check_predictions = check_logits.argmax(dim=1).tolist()
        tiny_history.append((tiny_step, check_loss, check_predictions))
        print(f'Step {tiny_step:3d}: loss={check_loss:.4f}, predictions={check_predictions}')
        if check_predictions == tiny_labels_gpu.tolist():
            break
torch.cuda.synchronize()
print(f'Tiny-set check: {tiny_step} steps, {time.perf_counter() - tiny_start:.1f} s')
print('Both examples learned:', check_predictions == tiny_labels_gpu.tolist())
print('Detector remains frozen:', all(not p.requires_grad for p in integrated_model.parameters()))
print('No validation or test images were used.')

**Checkpoint/resume smoke test.** Save the tiny-set class vectors and optimizer state outside Git, reload them into fresh objects, and verify that the next update matches an uninterrupted update. This is not a full-run checkpoint.

In [ ]:
# CELL 21
checkpoint_dir = DATA_ROOT / 'yolo_clip_checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
smoke_checkpoint = checkpoint_dir / f'task4_tiny_roundtrip_{time.time_ns()}.pt'
smoke_state = {
    'kind': 'task4_tiny_roundtrip',
    'completed_epochs': 0,
    'global_step': tiny_step,
    'seed': TRAIN_RECIPE['seed'],
    'starting_checkpoint_sha256': STARTING_SHA256,
    'animal_class_ids': animal_class_ids,
    'animal_prompts': animal_prompts,
    'recipe': dict(TRAIN_RECIPE),
    'class_vectors': tiny_vectors.detach().cpu().clone(),
    'optimizer_state': tiny_optimizer.state_dict(),
    'torch_rng_state': torch.get_rng_state(),
    'cuda_rng_states': torch.cuda.get_rng_state_all(),
    'loader_generator_state': loader_generator.get_state(),
}
torch.save(smoke_state, smoke_checkpoint)
loaded_state = torch.load(smoke_checkpoint, map_location='cpu', weights_only=True)
assert loaded_state['kind'] == 'task4_tiny_roundtrip'
assert loaded_state['starting_checkpoint_sha256'] == STARTING_SHA256
assert tuple(loaded_state['animal_prompts']) == animal_prompts
assert loaded_state['recipe'] == TRAIN_RECIPE
assert torch.equal(loaded_state['class_vectors'], tiny_vectors.detach().cpu())
resumed_vectors = torch.nn.Parameter(
    loaded_state['class_vectors'].to(device=probe_tensor.device).clone()
)
resumed_optimizer = torch.optim.AdamW(
    [resumed_vectors], lr=TRAIN_RECIPE['learning_rate'],
    weight_decay=TRAIN_RECIPE['weight_decay'],
)
resumed_optimizer.load_state_dict(loaded_state['optimizer_state'])
for state_key in ('step', 'exp_avg', 'exp_avg_sq'):
    assert torch.equal(
        tiny_optimizer.state[tiny_vectors][state_key].detach().cpu(),
        resumed_optimizer.state[resumed_vectors][state_key].detach().cpu(),
    ), f'Optimizer state differs: {state_key}'
def one_tiny_resume_step(vectors, optimizer):
    optimizer.zero_grad(set_to_none=True)
    output = integrated_model.predict(tiny_images_gpu, txt_feats=vectors)
    loss = F.cross_entropy(image_level_logits(output[1]['scores']), tiny_labels_gpu)
    loss.backward()
    torch.nn.utils.clip_grad_norm_([vectors], TRAIN_RECIPE['gradient_clip_norm'])
    optimizer.step()
    return float(loss.detach().item())
uninterrupted_loss = one_tiny_resume_step(tiny_vectors, tiny_optimizer)
resumed_loss = one_tiny_resume_step(resumed_vectors, resumed_optimizer)
max_resume_difference = float(
    (tiny_vectors.detach() - resumed_vectors.detach()).abs().max().item()
)
assert abs(uninterrupted_loss - resumed_loss) < 1e-5
assert max_resume_difference < 1e-6, 'Resumed update diverged'
print('Smoke checkpoint:', smoke_checkpoint)
print('Bytes:', smoke_checkpoint.stat().st_size)
print('Saved step:', loaded_state['global_step'], '| completed epochs:', loaded_state['completed_epochs'])
print(f'Next-step loss: uninterrupted={uninterrupted_loss:.6f}, resumed={resumed_loss:.6f}')
print('Maximum vector difference after next step:', max_resume_difference)
print('Checkpoint/resume roundtrip passed; no full training run started.')

## Task 5: train the first baseline

This is a fresh run from the original CLIP class vectors, not the smoke-test vectors. First check the run directory and data loaders; the next cells will perform training and save epoch-boundary checkpoints. The official `val` split remains reserved for later threshold tuning.

In [ ]:
# CELL 22
run_id = f'yoloclip_seed{TRAIN_RECIPE["seed"]}_{time.time_ns()}'
run_dir = checkpoint_dir / run_id
run_dir.mkdir(parents=True, exist_ok=False)
baseline_vectors = torch.nn.Parameter(
    wilds_text_features.detach().to(device=probe_tensor.device, dtype=torch.float32).clone()
)
baseline_optimizer = torch.optim.AdamW(
    [baseline_vectors], lr=TRAIN_RECIPE['learning_rate'],
    weight_decay=TRAIN_RECIPE['weight_decay'],
)
id_val_image_data = WILDSImageLabels(
    dataset.get_subset('id_val'), TRAIN_RECIPE['image_size']
)
id_val_loader = DataLoader(
    id_val_image_data, batch_size=TRAIN_RECIPE['batch_size'],
    shuffle=False, num_workers=0, pin_memory=True,
)
def make_train_epoch_loader(epoch):
    assert 1 <= epoch <= TRAIN_RECIPE['max_epochs']
    epoch_generator = torch.Generator().manual_seed(TRAIN_RECIPE['seed'] + epoch)
    return DataLoader(
        train_image_data, batch_size=TRAIN_RECIPE['batch_size'],
        shuffle=True, num_workers=0, pin_memory=True,
        generator=epoch_generator,
    )
assert len(train_image_data) == expected_counts['train']
assert len(id_val_image_data) == expected_counts['id_val']
assert torch.equal(baseline_vectors.detach().cpu(), wilds_text_features.detach().cpu())
assert all(not p.requires_grad for p in integrated_model.parameters())
print('Run directory (outside Git):', run_dir)
print('Fresh trainable class-vector values:', baseline_vectors.numel())
print('Train images:', len(train_image_data), '| batches per epoch:', len(make_train_epoch_loader(1)))
print('Checkpoint-selection images (id_val):', len(id_val_image_data),
      '| batches:', len(id_val_loader))
print('Planned epochs:', TRAIN_RECIPE['max_epochs'])
print('No training or validation inference started in this cell.')

**W&B setup.** This notebook targets `team-support-vectors-idlf26/iwilds` on CoreWeave Forge. CELL 23 installs W&B and defines logging helpers. In CELL 24, replace the single API-key placeholder in your private Molab copy, then run it to log in. Do not share or commit a copy containing the real key.

In [ ]:
# CELL 23
import os

WANDB_FORGE_API_URL = 'https://api.forge.coreweave.com'
WANDB_TEAM = 'team-support-vectors-idlf26'
WANDB_PROJECT_NAME = 'iwilds'
os.environ['WANDB_BASE_URL'] = WANDB_FORGE_API_URL

try:
    import wandb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'wandb'])
    import wandb

print('W&B package:', metadata.version('wandb'))
print('W&B API endpoint:', WANDB_FORGE_API_URL)
print('W&B destination:', f'{WANDB_TEAM}/{WANDB_PROJECT_NAME}')
print('No remote W&B run was created in this cell.')

def start_baseline_wandb_run():
    if wandb.run is not None:
        if wandb.run.id != run_id:
            raise RuntimeError('A different W&B run is active; finish it before starting this baseline')
        return wandb.run
    run_config = {
        'model': 'YOLO-World S-v2 with trainable CLIP-initialized class vectors',
        'dataset': 'iWildCam-WILDS v2.0',
        'starting_checkpoint_sha256': STARTING_SHA256,
        'class_names': list(animal_prompts),
        'empty_class_id': 0,
        'train_split': 'train',
        'checkpoint_selection_split': 'id_val',
        'threshold_selection_split': 'val',
        **TRAIN_RECIPE,
    }
    run = wandb.init(
        project=WANDB_PROJECT_NAME, entity=WANDB_TEAM, id=run_id, resume='allow',
        name=f'yolo-clip-iwildcam-seed{TRAIN_RECIPE["seed"]}',
        group='yolo-clip-trained-baseline', job_type='train',
        tags=['iwildcam', 'image-level', 'class-vectors'],
        config=run_config, dir=str(run_dir), mode='online', save_code=False,
        settings=wandb.Settings(disable_code=True, disable_git=True, console='off'),
    )
    run.define_metric('global_step')
    run.define_metric('train/*', step_metric='global_step')
    run.define_metric('id_val/*', step_metric='global_step')
    return run

def log_baseline_batch(run, global_step, epoch, mean_loss, gradient_norm, images_seen, elapsed_seconds):
    run.log({
        'global_step': global_step,
        'train/epoch': epoch,
        'train/loss': mean_loss,
        'train/gradient_norm': gradient_norm,
        'train/learning_rate': baseline_optimizer.param_groups[0]['lr'],
        'train/images_seen': images_seen,
        'train/elapsed_seconds': elapsed_seconds,
    })

def log_baseline_epoch(run, global_step, epoch, train_loss, id_val_loss, checkpoint_path):
    run.log({
        'global_step': global_step,
        'train/epoch': epoch,
        'train/epoch_loss': train_loss,
        'id_val/cross_entropy': id_val_loss,
        'train/peak_gpu_gib': torch.cuda.max_memory_allocated() / gib,
        'train/checkpoint_path': str(checkpoint_path),
    })

**W&B login.** Edit only the placeholder in CELL 24. The key is not printed, but it will be present in that copy of the notebook source until you remove it.

In [ ]:
# CELL 24
WANDB_API_KEY = 'PASTE_YOUR_WANDB_API_KEY_HERE'

assert WANDB_API_KEY != 'PASTE_YOUR_WANDB_API_KEY_HERE', 'Replace the placeholder in CELL 24'
wandb_authenticated = wandb.login(key=WANDB_API_KEY)
print('W&B login completed:', wandb_authenticated)

**W&B run check.** CELL 25 finishes the earlier empty `baseline_...` run if it is still active, then opens the `yoloclip_...` run in the team project. If console capture is off, it finishes and resumes that **same run ID** with automatic stdout/stderr capture enabled for subsequent training. Code and Git capture remain disabled. It does not train; do not print your API key or share the filled-in CELL 24.

In [ ]:
# CELL 25
# W&B returns False from login() when a run is already active; that is not a failed login.
if not wandb_authenticated:
    assert wandb.run is not None, 'No active W&B run or confirmed login; rerun CELL 24'
    print('Using the existing authenticated W&B run; CELL 24 re-ran while it was active.')
if wandb.run is not None and wandb.run.id != run_id:
    assert wandb.run.id.startswith('baseline_seed'), 'Unexpected active W&B run'
    print('Finishing earlier empty run:', wandb.run.id)
    wandb.run.finish()
baseline_wandb_run = start_baseline_wandb_run()
if baseline_wandb_run.settings.console == 'off':
    print('Resuming the same W&B run with automatic console capture enabled.')
    baseline_wandb_run.finish()
    baseline_wandb_run = wandb.init(
        project=WANDB_PROJECT_NAME, entity=WANDB_TEAM, id=run_id, resume='must',
        name=f'yolo-clip-iwildcam-seed{TRAIN_RECIPE["seed"]}',
        group='yolo-clip-trained-baseline', job_type='train',
        dir=str(run_dir), mode='online', save_code=False,
        settings=wandb.Settings(
            disable_code=True, disable_git=True, console='wrap',
            console_multipart=True, console_chunk_max_seconds=30,
        ),
    )
    baseline_wandb_run.define_metric('global_step')
    baseline_wandb_run.define_metric('train/*', step_metric='global_step')
    baseline_wandb_run.define_metric('id_val/*', step_metric='global_step')
assert baseline_wandb_run.project == WANDB_PROJECT_NAME
assert baseline_wandb_run.entity == WANDB_TEAM
assert baseline_wandb_run.id == run_id
assert baseline_wandb_run.settings.disable_code
assert baseline_wandb_run.settings.console in ('wrap', 'wrap_raw', 'wrap_emu')
print('W&B run path:', baseline_wandb_run.path)
print('W&B run URL:', baseline_wandb_run.url)
print('Code capture disabled:', baseline_wandb_run.settings.disable_code)
print('Console capture:', baseline_wandb_run.settings.console)
print('No additional training started in this cell.')

**Task 5 pause point.** Share CELL 25 output and confirm that the run URL opens under the intended team project. Then we will add the first-epoch training loop and checkpoint upload.

**Task 5: first full epoch.** CELL 26 trains on `train` and computes the predeclared checkpoint-selection cross-entropy on `id_val`. It saves local progress every 500 batches and at the end of the epoch; rerunning the cell resumes from the last saved batch. The first run may take a while. Do not rerun CELL 22, which would create a different run ID and reset the vectors. We will verify this output before adding the checkpoint upload and later epochs.

In [ ]:
# CELL 26
from tqdm.auto import tqdm

EPOCH_ONE = 1
PROGRESS_EVERY_BATCHES = 500
epoch_one_progress_path = run_dir / 'epoch_01_progress.pt'
epoch_one_checkpoint_path = run_dir / 'epoch_01.pt'

def save_epoch_one_state(path, batch_index, global_step, loss_total, examples_seen,
                         elapsed_seconds, id_val_loss=None):
    state = {
        'kind': 'yoloclip_image_level_baseline',
        'run_id': run_id,
        'epoch': EPOCH_ONE,
        'batch_in_epoch': batch_index,
        'global_step': global_step,
        'train_loss_total': loss_total,
        'train_examples_seen': examples_seen,
        'elapsed_seconds': elapsed_seconds,
        'id_val_cross_entropy': id_val_loss,
        'starting_checkpoint_sha256': STARTING_SHA256,
        'animal_class_ids': animal_class_ids,
        'animal_prompts': animal_prompts,
        'recipe': dict(TRAIN_RECIPE),
        'class_vectors': baseline_vectors.detach().cpu().clone(),
        'optimizer_state': baseline_optimizer.state_dict(),
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_states': torch.cuda.get_rng_state_all(),
    }
    temporary_path = path.with_suffix('.tmp')
    torch.save(state, temporary_path)
    os.replace(temporary_path, path)

def restore_epoch_one_state(path):
    state = torch.load(path, map_location='cpu', weights_only=True)
    assert state['kind'] == 'yoloclip_image_level_baseline'
    assert state['run_id'] == run_id
    assert state['epoch'] == EPOCH_ONE
    assert state['starting_checkpoint_sha256'] == STARTING_SHA256
    assert tuple(state['animal_class_ids']) == animal_class_ids
    assert tuple(state['animal_prompts']) == animal_prompts
    assert state['recipe'] == TRAIN_RECIPE
    with torch.no_grad():
        baseline_vectors.copy_(state['class_vectors'].to(baseline_vectors.device))
    baseline_optimizer.load_state_dict(state['optimizer_state'])
    return state

def train_and_validate_epoch_one():
    assert baseline_wandb_run.id == run_id and wandb.run is baseline_wandb_run
    assert all(not parameter.requires_grad for parameter in integrated_model.parameters())
    integrated_model.eval()  # Frozen detector and batch-normalization state.
    total_batches = len(make_train_epoch_loader(EPOCH_ONE))
    if epoch_one_checkpoint_path.is_file():
        completed = restore_epoch_one_state(epoch_one_checkpoint_path)
        assert completed['id_val_cross_entropy'] is not None
        print('Epoch 1 already complete; restored:', epoch_one_checkpoint_path)
        return epoch_one_checkpoint_path, completed['id_val_cross_entropy']
    if epoch_one_progress_path.is_file():
        saved = restore_epoch_one_state(epoch_one_progress_path)
        start_batch = saved['batch_in_epoch'] + 1
        global_step = saved['global_step']
        loss_total = saved['train_loss_total']
        examples_seen = saved['train_examples_seen']
        elapsed_before = saved['elapsed_seconds']
        print(f'Resuming epoch 1 after batch {start_batch - 1:,}/{total_batches:,}')
    else:
        start_batch, global_step = 1, 0
        loss_total, examples_seen, elapsed_before = 0.0, 0, 0.0
    assert 1 <= start_batch <= total_batches + 1
    assert global_step == start_batch - 1
    interval_loss, interval_examples = 0.0, 0
    call_started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    if start_batch <= total_batches:
        loader = make_train_epoch_loader(EPOCH_ONE)
        for batch_index, (images_cpu, labels_cpu, _) in enumerate(
            tqdm(loader, desc='Epoch 1 train', mininterval=10), start=1
        ):
            if batch_index < start_batch:
                continue  # Reconstruct the deterministic shuffle after an interruption.
            images = images_cpu.to(baseline_vectors.device, non_blocking=True)
            labels = labels_cpu.to(baseline_vectors.device, non_blocking=True)
            baseline_optimizer.zero_grad(set_to_none=True)
            output = integrated_model.predict(images, txt_feats=baseline_vectors)
            logits = image_level_logits(output[1]['scores'])
            loss = F.cross_entropy(logits, labels)
            assert torch.isfinite(loss), f'Non-finite loss at batch {batch_index}'
            loss.backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(
                [baseline_vectors], TRAIN_RECIPE['gradient_clip_norm']
            )
            assert torch.isfinite(gradient_norm), f'Non-finite gradient at batch {batch_index}'
            baseline_optimizer.step()
            batch_size = len(labels)
            weighted_loss = float(loss.detach().item()) * batch_size
            loss_total += weighted_loss
            examples_seen += batch_size
            interval_loss += weighted_loss
            interval_examples += batch_size
            global_step += 1
            if batch_index % 100 == 0 or batch_index == total_batches:
                elapsed = elapsed_before + time.perf_counter() - call_started
                log_baseline_batch(
                    baseline_wandb_run, global_step, EPOCH_ONE,
                    interval_loss / interval_examples, float(gradient_norm.item()),
                    examples_seen, elapsed,
                )
                interval_loss, interval_examples = 0.0, 0
            if batch_index % PROGRESS_EVERY_BATCHES == 0 or batch_index == total_batches:
                elapsed = elapsed_before + time.perf_counter() - call_started
                save_epoch_one_state(
                    epoch_one_progress_path, batch_index, global_step,
                    loss_total, examples_seen, elapsed,
                )
                print(f'Saved batch {batch_index:,}/{total_batches:,}; '
                      f'mean train loss={loss_total / examples_seen:.4f}')
    assert examples_seen == len(train_image_data)
    assert global_step == total_batches
    train_loss = loss_total / examples_seen
    val_loss_total, val_examples = 0.0, 0
    with torch.no_grad():
        for images_cpu, labels_cpu, _ in tqdm(
            id_val_loader, desc='Epoch 1 id_val', mininterval=10
        ):
            images = images_cpu.to(baseline_vectors.device, non_blocking=True)
            labels = labels_cpu.to(baseline_vectors.device, non_blocking=True)
            output = integrated_model.predict(images, txt_feats=baseline_vectors)
            logits = image_level_logits(output[1]['scores'])
            val_loss = F.cross_entropy(logits, labels)
            assert torch.isfinite(val_loss), 'Non-finite id_val loss'
            val_loss_total += float(val_loss.item()) * len(labels)
            val_examples += len(labels)
    assert val_examples == len(id_val_image_data)
    id_val_cross_entropy = val_loss_total / val_examples
    elapsed = elapsed_before + time.perf_counter() - call_started
    save_epoch_one_state(
        epoch_one_checkpoint_path, total_batches, global_step,
        loss_total, examples_seen, elapsed, id_val_cross_entropy,
    )
    log_baseline_epoch(
        baseline_wandb_run, global_step, EPOCH_ONE, train_loss,
        id_val_cross_entropy, epoch_one_checkpoint_path,
    )
    print(f'Epoch 1 complete: train CE={train_loss:.4f}; '
          f'id_val CE={id_val_cross_entropy:.4f}; {elapsed / 60:.1f} min')
    print('Checkpoint:', epoch_one_checkpoint_path)
    print('Peak PyTorch GPU allocation:',
          f'{torch.cuda.max_memory_allocated() / gib:.2f} GiB')
    print('No held-out-camera validation or test split used.')
    return epoch_one_checkpoint_path, id_val_cross_entropy

first_epoch_checkpoint, first_epoch_id_val_loss = train_and_validate_epoch_one()

**Protect the completed checkpoints.** CELL 27 uploads only the small training checkpoint (class vectors, optimizer, and recipe) to this W&B run as a model artifact. It does not upload the dataset, notebook, or API key. The upload is idempotent within this Molab directory.

In [ ]:
# CELL 27
ARTIFACT_COLLECTION_NAME = f'yoloclip-checkpoints-{run_id}'

def upload_yoloclip_checkpoint(epoch, checkpoint_path):
    assert baseline_wandb_run.id == run_id and wandb.run is baseline_wandb_run
    assert checkpoint_path.is_file(), f'Missing checkpoint: {checkpoint_path}'
    uploaded_marker = run_dir / f'epoch_{epoch:02d}_uploaded.txt'
    if uploaded_marker.is_file():
        print(f'Epoch {epoch} already uploaded:', uploaded_marker.read_text().strip())
        return
    artifact = wandb.Artifact(
        ARTIFACT_COLLECTION_NAME, type='model',
        metadata={'epoch': epoch, 'run_id': run_id,
                  'starting_checkpoint_sha256': STARTING_SHA256},
    )
    artifact.add_file(str(checkpoint_path), name=checkpoint_path.name)
    uploaded = baseline_wandb_run.log_artifact(
        artifact, aliases=[f'epoch-{epoch}', 'latest']
    ).wait()
    uploaded_marker.write_text(uploaded.name + '\n')
    print(f'Epoch {epoch} checkpoint uploaded:', uploaded.name)
    print('Checkpoint bytes:', checkpoint_path.stat().st_size)

epoch_one_state = restore_epoch_one_state(epoch_one_checkpoint_path)
assert epoch_one_state['id_val_cross_entropy'] is not None
assert epoch_one_state['batch_in_epoch'] == len(make_train_epoch_loader(1))
upload_yoloclip_checkpoint(1, epoch_one_checkpoint_path)

**Epoch-1 inference sanity check.** CELL 28 restores the saved epoch-1 vectors and predicts a fixed, balanced sample of 16 `id_val` images using the same *pre-NMS max-region plus empty-logit* rule used in training. This is a diagnostic, not the final post-NMS/validation-threshold inference rule. Do not treat the tiny sample's accuracy as a benchmark result or use it to change the recipe.

In [ ]:
# CELL 28
import matplotlib.pyplot as plt

assert (run_dir / 'epoch_01_uploaded.txt').is_file(), 'Run CELL 27 first'
diagnostic_state = restore_epoch_one_state(epoch_one_checkpoint_path)
assert diagnostic_state['id_val_cross_entropy'] is not None
integrated_model.eval()
diagnostic_labels = id_val_image_data.subset.y_array
diagnostic_empty_indices = torch.nonzero(diagnostic_labels == 0).flatten()
diagnostic_animal_indices = torch.nonzero(diagnostic_labels != 0).flatten()
diagnostic_generator = torch.Generator().manual_seed(TRAIN_RECIPE['seed'] + 1000)
diagnostic_indices = torch.cat((
    diagnostic_empty_indices[torch.randperm(
        len(diagnostic_empty_indices), generator=diagnostic_generator)[:8]],
    diagnostic_animal_indices[torch.randperm(
        len(diagnostic_animal_indices), generator=diagnostic_generator)[:8]],
)).tolist()
diagnostic_loader = DataLoader(
    Subset(id_val_image_data, diagnostic_indices), batch_size=16, shuffle=False,
    num_workers=0, pin_memory=True,
)
diagnostic_images_cpu, diagnostic_targets_cpu, _ = next(iter(diagnostic_loader))
with torch.no_grad():
    diagnostic_output = integrated_model.predict(
        diagnostic_images_cpu.to(baseline_vectors.device, non_blocking=True),
        txt_feats=baseline_vectors,
    )
    diagnostic_logits = image_level_logits(diagnostic_output[1]['scores'])
    diagnostic_predictions = diagnostic_logits.argmax(dim=1).cpu()
assert tuple(diagnostic_logits.shape) == (16, dataset.n_classes)
assert diagnostic_targets_cpu[:8].eq(0).all()
assert diagnostic_targets_cpu[8:].ne(0).all()
print('Fixed id_val sample: 8 empty + 8 animal images; no threshold tuning')
print('Maximum change from initial CLIP vectors:',
      float((baseline_vectors.detach().cpu() - wilds_text_features.detach().cpu()).abs().max()))
print('index | true label -> predicted label | top raw logit | correct')
for diagnostic_row, dataset_index in enumerate(diagnostic_indices):
    true_y = int(diagnostic_targets_cpu[diagnostic_row])
    predicted_y = int(diagnostic_predictions[diagnostic_row])
    top_logit = float(diagnostic_logits[diagnostic_row, predicted_y].item())
    print(f'{dataset_index:5d} | {true_y:3d} {id_to_name[true_y]} -> '
          f'{predicted_y:3d} {id_to_name[predicted_y]} | '
          f'{top_logit:+.3f} | {true_y == predicted_y}')
diagnostic_correct = int(diagnostic_predictions.eq(diagnostic_targets_cpu).sum())
print(f'Sanity sample correct: {diagnostic_correct}/16 (not a benchmark metric)')
figure, axes = plt.subplots(2, 2, figsize=(11, 9))
for axis, diagnostic_row in zip(axes.flat, (0, 1, 8, 9)):
    axis.imshow(diagnostic_images_cpu[diagnostic_row].permute(1, 2, 0).numpy())
    true_y = int(diagnostic_targets_cpu[diagnostic_row])
    predicted_y = int(diagnostic_predictions[diagnostic_row])
    axis.set_title(f'True: {id_to_name[true_y]}\nPred: {id_to_name[predicted_y]}')
    axis.axis('off')
figure.tight_layout()
plt.show()
print('No held-out-camera validation or test split used.')

**Finish the five-epoch baseline.** After the CELL 28 sanity check, run CELLS 29–32 in order. Each cell trains exactly one additional epoch, computes `id_val` cross-entropy, saves a resumable local checkpoint every 500 batches, and uploads the completed epoch. If a cell is interrupted while `/marimo/data` remains available, rerun that same cell. Do not rerun CELL 22 or change the recipe during this run.

In [ ]:
# CELL 29
def save_later_epoch_state(path, epoch, batch_index, global_step, loss_total,
                           examples_seen, elapsed_seconds, id_val_loss=None):
    state = {
        'kind': 'yoloclip_image_level_baseline',
        'run_id': run_id,
        'epoch': epoch,
        'batch_in_epoch': batch_index,
        'global_step': global_step,
        'train_loss_total': loss_total,
        'train_examples_seen': examples_seen,
        'elapsed_seconds': elapsed_seconds,
        'id_val_cross_entropy': id_val_loss,
        'starting_checkpoint_sha256': STARTING_SHA256,
        'animal_class_ids': animal_class_ids,
        'animal_prompts': animal_prompts,
        'recipe': dict(TRAIN_RECIPE),
        'class_vectors': baseline_vectors.detach().cpu().clone(),
        'optimizer_state': baseline_optimizer.state_dict(),
        'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_states': torch.cuda.get_rng_state_all(),
    }
    temporary_path = path.with_suffix('.tmp')
    torch.save(state, temporary_path)
    os.replace(temporary_path, path)

def restore_later_epoch_state(path, expected_epoch):
    state = torch.load(path, map_location='cpu', weights_only=True)
    assert state['kind'] == 'yoloclip_image_level_baseline'
    assert state['run_id'] == run_id and state['epoch'] == expected_epoch
    assert state['starting_checkpoint_sha256'] == STARTING_SHA256
    assert tuple(state['animal_class_ids']) == animal_class_ids
    assert tuple(state['animal_prompts']) == animal_prompts
    assert state['recipe'] == TRAIN_RECIPE
    with torch.no_grad():
        baseline_vectors.copy_(state['class_vectors'].to(baseline_vectors.device))
    baseline_optimizer.load_state_dict(state['optimizer_state'])
    return state

def run_later_epoch(epoch):
    assert 2 <= epoch <= TRAIN_RECIPE['max_epochs']
    assert baseline_wandb_run.id == run_id and wandb.run is baseline_wandb_run
    assert (run_dir / 'epoch_01_uploaded.txt').is_file(), 'Run CELL 27 first'
    assert all(not parameter.requires_grad for parameter in integrated_model.parameters())
    integrated_model.eval()
    previous_path = run_dir / f'epoch_{epoch - 1:02d}.pt'
    completed_path = run_dir / f'epoch_{epoch:02d}.pt'
    assert previous_path.is_file(), f'Missing prior epoch checkpoint: {previous_path}'
    if epoch == 2:
        previous = restore_epoch_one_state(previous_path)
    else:
        previous = restore_later_epoch_state(previous_path, epoch - 1)
    assert previous['id_val_cross_entropy'] is not None
    previous_global_step = previous['global_step']
    total_batches = len(make_train_epoch_loader(epoch))
    if completed_path.is_file():
        completed = restore_later_epoch_state(completed_path, epoch)
        assert completed['id_val_cross_entropy'] is not None
        upload_yoloclip_checkpoint(epoch, completed_path)
        print(f'Epoch {epoch} already complete; restored:', completed_path)
        return completed_path, completed['id_val_cross_entropy']
    global_step = previous_global_step
    loss_total, examples_seen = 0.0, 0
    interval_loss, interval_examples = 0.0, 0
    call_started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    loader = make_train_epoch_loader(epoch)
    for batch_index, (images_cpu, labels_cpu, _) in enumerate(
        tqdm(loader, desc=f'Epoch {epoch} train', mininterval=10), start=1
    ):
        images = images_cpu.to(baseline_vectors.device, non_blocking=True)
        labels = labels_cpu.to(baseline_vectors.device, non_blocking=True)
        baseline_optimizer.zero_grad(set_to_none=True)
        output = integrated_model.predict(images, txt_feats=baseline_vectors)
        logits = image_level_logits(output[1]['scores'])
        loss = F.cross_entropy(logits, labels)
        assert torch.isfinite(loss), f'Non-finite loss at epoch {epoch}, batch {batch_index}'
        loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            [baseline_vectors], TRAIN_RECIPE['gradient_clip_norm']
        )
        assert torch.isfinite(gradient_norm), f'Non-finite gradient at batch {batch_index}'
        baseline_optimizer.step()
        batch_size = len(labels)
        weighted_loss = float(loss.detach().item()) * batch_size
        loss_total += weighted_loss
        examples_seen += batch_size
        interval_loss += weighted_loss
        interval_examples += batch_size
        global_step += 1
        if batch_index % 100 == 0 or batch_index == total_batches:
            elapsed = time.perf_counter() - call_started
            log_baseline_batch(
                baseline_wandb_run, global_step, epoch,
                interval_loss / interval_examples, float(gradient_norm.item()),
                examples_seen, elapsed,
            )
            interval_loss, interval_examples = 0.0, 0
        if batch_index % PROGRESS_EVERY_BATCHES == 0 or batch_index == total_batches:
            print(f'Epoch {epoch} batch {batch_index:,}/{total_batches:,}; '
                  f'mean train loss={loss_total / examples_seen:.4f}')
    assert examples_seen == len(train_image_data)
    assert global_step == previous_global_step + total_batches
    train_loss = loss_total / examples_seen
    val_loss_total, val_examples = 0.0, 0
    with torch.no_grad():
        for images_cpu, labels_cpu, _ in tqdm(
            id_val_loader, desc=f'Epoch {epoch} id_val', mininterval=10
        ):
            images = images_cpu.to(baseline_vectors.device, non_blocking=True)
            labels = labels_cpu.to(baseline_vectors.device, non_blocking=True)
            output = integrated_model.predict(images, txt_feats=baseline_vectors)
            logits = image_level_logits(output[1]['scores'])
            val_loss = F.cross_entropy(logits, labels)
            assert torch.isfinite(val_loss), 'Non-finite id_val loss'
            val_loss_total += float(val_loss.item()) * len(labels)
            val_examples += len(labels)
    assert val_examples == len(id_val_image_data)
    id_val_cross_entropy = val_loss_total / val_examples
    elapsed = time.perf_counter() - call_started
    save_later_epoch_state(
        completed_path, epoch, total_batches, global_step,
        loss_total, examples_seen, elapsed, id_val_cross_entropy,
    )
    log_baseline_epoch(
        baseline_wandb_run, global_step, epoch, train_loss,
        id_val_cross_entropy, completed_path,
    )
    print(f'Epoch {epoch} complete: train CE={train_loss:.4f}; '
          f'id_val CE={id_val_cross_entropy:.4f}; {elapsed / 60:.1f} min')
    print('Checkpoint:', completed_path)
    print('Peak PyTorch GPU allocation:',
          f'{torch.cuda.max_memory_allocated() / gib:.2f} GiB')
    upload_yoloclip_checkpoint(epoch, completed_path)
    print('No held-out-camera validation or test split used.')
    return completed_path, id_val_cross_entropy

epoch_two_result = run_later_epoch(2)

In [ ]:
# CELL 30
assert epoch_two_result[0].is_file(), 'Complete CELL 28 first'
epoch_three_result = run_later_epoch(3)

In [ ]:
# CELL 31
assert epoch_three_result[0].is_file(), 'Complete CELL 29 first'
epoch_four_result = run_later_epoch(4)

In [ ]:
# CELL 32
assert epoch_four_result[0].is_file(), 'Complete CELL 30 first'
epoch_five_result = run_later_epoch(5)

**Continue training through epoch 20.** CELL 33 starts from the completed epoch-5 checkpoint, retaining the learned class vectors and AdamW state. It runs epochs 6–20 in one invocation, validates on `id_val` after each epoch, and saves/uploads one checkpoint per completed epoch. If interrupted, rerun CELL 33: completed epochs are skipped and the interrupted epoch restarts from the preceding epoch-end checkpoint. The original five-epoch recipe remains unchanged; extension checkpoints record the 20-epoch target. Do not rerun CELL 22 or use the held-out-camera validation/test splits here.

In [ ]:
# CELL 33
def train_epochs_six_through_twenty():
    target_epoch = 20
    assert TRAIN_RECIPE['max_epochs'] == 5, 'Keep the original recipe unchanged'
    assert epoch_five_result[0] == run_dir / 'epoch_05.pt'
    assert epoch_five_result[0].is_file(), 'Complete CELL 32 first'
    assert baseline_wandb_run.id == run_id and wandb.run is baseline_wandb_run
    assert all(not p.requires_grad for p in integrated_model.parameters())
    integrated_model.eval()
    extended_recipe = dict(TRAIN_RECIPE)
    extended_recipe['max_epochs'] = target_epoch
    expected_recipe_without_limit = {
        key: value for key, value in TRAIN_RECIPE.items() if key != 'max_epochs'
    }

    def load_completed_epoch(epoch):
        path = run_dir / f'epoch_{epoch:02d}.pt'
        state = torch.load(path, map_location='cpu', weights_only=True)
        assert state['kind'] == 'yoloclip_image_level_baseline'
        assert state['run_id'] == run_id and state['epoch'] == epoch
        assert state['starting_checkpoint_sha256'] == STARTING_SHA256
        assert tuple(state['animal_class_ids']) == animal_class_ids
        assert tuple(state['animal_prompts']) == animal_prompts
        saved_recipe = dict(state['recipe'])
        assert saved_recipe.pop('max_epochs') in (5, target_epoch)
        assert saved_recipe == expected_recipe_without_limit
        assert state['id_val_cross_entropy'] is not None
        assert state['batch_in_epoch'] == (
            len(train_image_data) + TRAIN_RECIPE['batch_size'] - 1
        ) // TRAIN_RECIPE['batch_size']
        with torch.no_grad():
            baseline_vectors.copy_(state['class_vectors'].to(baseline_vectors.device))
        baseline_optimizer.load_state_dict(state['optimizer_state'])
        return state

    results = []
    previous_state = load_completed_epoch(5)
    for epoch in range(6, target_epoch + 1):
        checkpoint_path = run_dir / f'epoch_{epoch:02d}.pt'
        if checkpoint_path.is_file():
            previous_state = load_completed_epoch(epoch)
            upload_yoloclip_checkpoint(epoch, checkpoint_path)
            results.append((epoch, previous_state['id_val_cross_entropy']))
            print(f'Epoch {epoch} already complete; id_val CE='
                  f"{previous_state['id_val_cross_entropy']:.4f}")
            continue
        assert previous_state['epoch'] == epoch - 1
        total_batches = (len(train_image_data) + TRAIN_RECIPE['batch_size'] - 1) // (
            TRAIN_RECIPE['batch_size']
        )
        epoch_generator = torch.Generator().manual_seed(TRAIN_RECIPE['seed'] + epoch)
        loader = DataLoader(
            train_image_data, batch_size=TRAIN_RECIPE['batch_size'],
            shuffle=True, num_workers=0, pin_memory=True, generator=epoch_generator,
        )
        assert len(loader) == total_batches
        global_step = previous_state['global_step']
        loss_total, examples_seen = 0.0, 0
        interval_loss, interval_examples = 0.0, 0
        epoch_started = time.perf_counter()
        torch.cuda.reset_peak_memory_stats()
        for batch_index, (images_cpu, labels_cpu, _) in enumerate(
            tqdm(loader, desc=f'Epoch {epoch} train', mininterval=10), start=1
        ):
            images = images_cpu.to(baseline_vectors.device, non_blocking=True)
            labels = labels_cpu.to(baseline_vectors.device, non_blocking=True)
            baseline_optimizer.zero_grad(set_to_none=True)
            output = integrated_model.predict(images, txt_feats=baseline_vectors)
            logits = image_level_logits(output[1]['scores'])
            loss = F.cross_entropy(logits, labels)
            assert torch.isfinite(loss), f'Non-finite loss at epoch {epoch}, batch {batch_index}'
            loss.backward()
            gradient_norm = torch.nn.utils.clip_grad_norm_(
                [baseline_vectors], TRAIN_RECIPE['gradient_clip_norm']
            )
            assert torch.isfinite(gradient_norm)
            baseline_optimizer.step()
            batch_size = len(labels)
            weighted_loss = float(loss.detach().item()) * batch_size
            loss_total += weighted_loss
            examples_seen += batch_size
            interval_loss += weighted_loss
            interval_examples += batch_size
            global_step += 1
            if batch_index % 100 == 0 or batch_index == total_batches:
                log_baseline_batch(
                    baseline_wandb_run, global_step, epoch,
                    interval_loss / interval_examples, float(gradient_norm.item()),
                    examples_seen, time.perf_counter() - epoch_started,
                )
                interval_loss, interval_examples = 0.0, 0
            if batch_index % 500 == 0 or batch_index == total_batches:
                print(f'Epoch {epoch} batch {batch_index:,}/{total_batches:,}; '
                      f'mean train loss={loss_total / examples_seen:.4f}')
        assert examples_seen == len(train_image_data)
        assert global_step == previous_state['global_step'] + total_batches
        train_loss = loss_total / examples_seen
        val_loss_total, val_examples = 0.0, 0
        with torch.no_grad():
            for images_cpu, labels_cpu, _ in tqdm(
                id_val_loader, desc=f'Epoch {epoch} id_val', mininterval=10
            ):
                images = images_cpu.to(baseline_vectors.device, non_blocking=True)
                labels = labels_cpu.to(baseline_vectors.device, non_blocking=True)
                output = integrated_model.predict(images, txt_feats=baseline_vectors)
                logits = image_level_logits(output[1]['scores'])
                val_loss = F.cross_entropy(logits, labels)
                assert torch.isfinite(val_loss), 'Non-finite id_val loss'
                val_loss_total += float(val_loss.item()) * len(labels)
                val_examples += len(labels)
        assert val_examples == len(id_val_image_data)
        id_val_cross_entropy = val_loss_total / val_examples
        elapsed = time.perf_counter() - epoch_started
        state = {
            'kind': 'yoloclip_image_level_baseline',
            'run_id': run_id, 'epoch': epoch,
            'batch_in_epoch': total_batches, 'global_step': global_step,
            'train_loss_total': loss_total, 'train_examples_seen': examples_seen,
            'elapsed_seconds': elapsed,
            'id_val_cross_entropy': id_val_cross_entropy,
            'starting_checkpoint_sha256': STARTING_SHA256,
            'animal_class_ids': animal_class_ids,
            'animal_prompts': animal_prompts,
            'recipe': extended_recipe,
            'class_vectors': baseline_vectors.detach().cpu().clone(),
            'optimizer_state': baseline_optimizer.state_dict(),
            'torch_rng_state': torch.get_rng_state(),
            'cuda_rng_states': torch.cuda.get_rng_state_all(),
        }
        temporary_path = checkpoint_path.with_suffix('.tmp')
        torch.save(state, temporary_path)
        os.replace(temporary_path, checkpoint_path)
        log_baseline_epoch(
            baseline_wandb_run, global_step, epoch, train_loss,
            id_val_cross_entropy, checkpoint_path,
        )
        print(f'Epoch {epoch} complete: train CE={train_loss:.4f}; '
              f'id_val CE={id_val_cross_entropy:.4f}; {elapsed / 60:.1f} min')
        print('Checkpoint:', checkpoint_path)
        print('Peak PyTorch GPU allocation:',
              f'{torch.cuda.max_memory_allocated() / gib:.2f} GiB')
        upload_yoloclip_checkpoint(epoch, checkpoint_path)
        previous_state = state
        results.append((epoch, id_val_cross_entropy))
    early_results = []
    for epoch in range(1, 6):
        state = torch.load(
            run_dir / f'epoch_{epoch:02d}.pt', map_location='cpu', weights_only=True
        )
        assert state['run_id'] == run_id and state['epoch'] == epoch
        early_results.append((epoch, float(state['id_val_cross_entropy'])))
    best_epoch, best_loss = min(early_results + results, key=lambda item: item[1])
    print(f'Best checkpoint through epoch 20: epoch {best_epoch}, id_val CE={best_loss:.4f}')
    print('No held-out-camera validation or test split used.')
    return {'best_epoch': best_epoch, 'best_id_val_ce': best_loss, 'epochs_6_to_20': results}

extended_training_summary = train_epochs_six_through_twenty()

**Evaluate before the final test.** CELL 34 restores the lowest-`id_val`-CE checkpoint and smoke-tests post-NMS image-level inference on two `id_val` images. CELL 35 then runs the official held-out-camera `val` split, selects the empty threshold using its macro-F1, and freezes the checkpoint/rule/threshold in a manifest. Its `val` score is threshold-tuned, not independent test performance. Neither cell opens a test split. Share these outputs before we add the one-time test evaluation.

In [ ]:
# CELL 34
from ultralytics.utils import nms as trained_nms

EVAL_BEST_EPOCH = int(extended_training_summary['best_epoch'])
assert EVAL_BEST_EPOCH == 9, 'Review the selected epoch against CELL 33 output'
EVAL_CHECKPOINT = run_dir / f'epoch_{EVAL_BEST_EPOCH:02d}.pt'
EVAL_CHECKPOINT_SHA256 = hashlib.sha256(EVAL_CHECKPOINT.read_bytes()).hexdigest()
eval_checkpoint_state = torch.load(EVAL_CHECKPOINT, map_location='cpu', weights_only=True)
assert eval_checkpoint_state['run_id'] == run_id
assert eval_checkpoint_state['epoch'] == EVAL_BEST_EPOCH
assert eval_checkpoint_state['starting_checkpoint_sha256'] == STARTING_SHA256
assert tuple(eval_checkpoint_state['animal_class_ids']) == animal_class_ids
assert tuple(eval_checkpoint_state['animal_prompts']) == animal_prompts
assert float(eval_checkpoint_state['id_val_cross_entropy']) == (
    extended_training_summary['best_id_val_ce']
)
assert ultralytics.__version__ == '8.4.178', 'Review NMS behavior after version change'
with torch.no_grad():
    baseline_vectors.copy_(eval_checkpoint_state['class_vectors'].to(baseline_vectors.device))
integrated_model.eval()
EVAL_NMS_FLOOR = 0.001
EVAL_NMS_IOU = 0.7
EVAL_NMS_MAX_BOXES = 20
eval_smoke_images, eval_smoke_labels, _ = next(iter(id_val_loader))
with torch.inference_mode():
    eval_smoke_output = integrated_model.predict(
        eval_smoke_images[:2].to(baseline_vectors.device), txt_feats=baseline_vectors
    )
    eval_smoke_boxes = trained_nms.non_max_suppression(
        eval_smoke_output[0], conf_thres=EVAL_NMS_FLOOR,
        iou_thres=EVAL_NMS_IOU, max_det=EVAL_NMS_MAX_BOXES,
        nc=len(animal_class_ids),
    )
assert len(eval_smoke_boxes) == 2
for eval_smoke_i, eval_boxes in enumerate(eval_smoke_boxes):
    assert eval_boxes.ndim == 2 and eval_boxes.shape[1] >= 6
    if len(eval_boxes):
        assert torch.isfinite(eval_boxes).all()
        assert bool(((eval_boxes[:, 4] >= 0) & (eval_boxes[:, 4] <= 1)).all())
        assert bool(((eval_boxes[:, 5] >= 0) & (eval_boxes[:, 5] < 181)).all())
        eval_top = eval_boxes[eval_boxes[:, 4].argmax()]
        eval_pred_y = int(eval_top[5].item()) + 1
        eval_score = float(eval_top[4].item())
    else:
        eval_pred_y, eval_score = 0, -1.0
    print(f'id_val sample {eval_smoke_i}: true={int(eval_smoke_labels[eval_smoke_i])} '
          f'best-detection class={eval_pred_y}, score={eval_score:.4f}, '
          f'boxes={len(eval_boxes)}')
print('Selected epoch:', EVAL_BEST_EPOCH, '| id_val CE:',
      eval_checkpoint_state['id_val_cross_entropy'])
print('Checkpoint SHA-256:', EVAL_CHECKPOINT_SHA256)
print('Post-NMS smoke test passed; no threshold or test data used.')

In [ ]:
# CELL 35
def infer_trained_top_detections(split_name):
    import json

    assert split_name == 'val', 'Only validation inference is enabled in CELL 35'
    subset = dataset.get_subset(split_name)
    config = {
        'protocol': 'trained-yoloclip-post-nms-v1',
        'split': split_name, 'images': len(subset), 'run_id': run_id,
        'checkpoint_epoch': EVAL_BEST_EPOCH,
        'checkpoint_sha256': EVAL_CHECKPOINT_SHA256,
        'ultralytics_version': ultralytics.__version__,
        'image_size': TRAIN_RECIPE['image_size'],
        'nms_floor': EVAL_NMS_FLOOR, 'nms_iou': EVAL_NMS_IOU,
        'nms_max_boxes': EVAL_NMS_MAX_BOXES,
        'animal_class_ids': list(animal_class_ids),
        'animal_prompts': list(animal_prompts),
    }
    config_json = json.dumps(config, sort_keys=True)
    signature = hashlib.sha256(config_json.encode('utf-8')).hexdigest()[:16]
    cache_path = run_dir / f'epoch_{EVAL_BEST_EPOCH:02d}_{split_name}_{signature}.npz'
    if cache_path.is_file():
        with np.load(cache_path, allow_pickle=False) as cache:
            assert str(cache['config_json'].item()) == config_json
            records = {key: cache[key].copy() for key in (
                'true_y', 'location', 'top_y', 'top_score', 'box_count', 'raw_pred_y'
            )}
        print('Loaded existing validation inference:', cache_path)
    else:
        loader = DataLoader(
            WILDSImageLabels(subset, TRAIN_RECIPE['image_size']),
            batch_size=TRAIN_RECIPE['batch_size'], shuffle=False,
            num_workers=0, pin_memory=True,
        )
        location_index = dataset.metadata_fields.index('location')
        lists = {key: [] for key in (
            'true_y', 'location', 'top_y', 'top_score', 'box_count', 'raw_pred_y'
        )}
        integrated_model.eval()
        with torch.inference_mode():
            for images_cpu, labels_cpu, metadata_cpu in tqdm(
                loader, desc='Held-out-camera val inference', mininterval=10
            ):
                output = integrated_model.predict(
                    images_cpu.to(baseline_vectors.device), txt_feats=baseline_vectors
                )
                raw_pred = image_level_logits(output[1]['scores']).argmax(dim=1)
                boxes_per_image = trained_nms.non_max_suppression(
                    output[0], conf_thres=EVAL_NMS_FLOOR,
                    iou_thres=EVAL_NMS_IOU, max_det=EVAL_NMS_MAX_BOXES,
                    nc=len(animal_class_ids),
                )
                assert len(boxes_per_image) == len(labels_cpu)
                lists['true_y'].extend(labels_cpu.tolist())
                lists['location'].extend(metadata_cpu[:, location_index].tolist())
                lists['raw_pred_y'].extend(raw_pred.cpu().tolist())
                for boxes in boxes_per_image:
                    lists['box_count'].append(len(boxes))
                    if len(boxes):
                        top = boxes[boxes[:, 4].argmax()]
                        lists['top_y'].append(int(top[5].item()) + 1)
                        lists['top_score'].append(float(top[4].item()))
                    else:
                        lists['top_y'].append(0)
                        lists['top_score'].append(-1.0)
        records = {
            'true_y': np.asarray(lists['true_y'], dtype=np.int16),
            'location': np.asarray(lists['location'], dtype=np.int32),
            'top_y': np.asarray(lists['top_y'], dtype=np.int16),
            'top_score': np.asarray(lists['top_score'], dtype=np.float32),
            'box_count': np.asarray(lists['box_count'], dtype=np.uint8),
            'raw_pred_y': np.asarray(lists['raw_pred_y'], dtype=np.int16),
        }
        temporary_path = cache_path.with_suffix('.tmp')
        with temporary_path.open('wb') as stream:
            np.savez_compressed(stream, config_json=np.asarray(config_json), **records)
        os.replace(temporary_path, cache_path)
        print('Saved validation inference:', cache_path)
    assert all(len(values) == len(subset) for values in records.values())
    assert np.array_equal(records['true_y'], subset.y_array.numpy())
    assert np.array_equal(
        records['location'], subset.metadata_array[:, dataset.metadata_fields.index(
            'location'
        )].numpy()
    )
    assert np.all((records['top_y'] >= 0) & (records['top_y'] < dataset.n_classes))
    return subset, records, cache_path, config

def evaluate_and_freeze_validation():
    import json

    subset, records, cache_path, config = infer_trained_top_detections('val')
    true_y = records['true_y']
    top_y = records['top_y']
    top_score = records['top_score']
    positive_scores = top_score[top_score >= 0]
    candidates = {0.0, 1.0}
    candidates.update(float(x) for x in np.linspace(0.01, 0.99, 99))
    if len(positive_scores):
        candidates.update(float(x) for x in np.quantile(
            positive_scores, np.linspace(0, 1, 101)
        ))
    sweep = []
    for threshold in tqdm(sorted(candidates), desc='Threshold sweep', mininterval=10):
        predicted = np.where(top_score >= threshold, top_y, 0).astype(np.int64)
        metrics, _ = dataset.eval(
            torch.from_numpy(predicted.copy()), subset.y_array, subset.metadata_array
        )
        f1_keys = [key for key in metrics if key.startswith('F1-macro')]
        assert len(f1_keys) == 1, f'Unexpected WILDS metric keys: {list(metrics)}'
        sweep.append((float(threshold), float(metrics[f1_keys[0]])))
    selected_threshold, selected_f1 = max(sweep, key=lambda row: (row[1], -row[0]))
    selected_predictions = np.where(
        top_score >= selected_threshold, top_y, 0
    ).astype(np.int64)
    official_metrics, official_text = dataset.eval(
        torch.from_numpy(selected_predictions.copy()), subset.y_array,
        subset.metadata_array,
    )
    raw_metrics, raw_text = dataset.eval(
        torch.from_numpy(records['raw_pred_y'].astype(np.int64)),
        subset.y_array, subset.metadata_array,
    )
    assert abs(float(official_metrics[f1_keys[0]]) - selected_f1) < 1e-8
    manifest = {
        **config, 'validation_cache': str(cache_path),
        'threshold_objective': 'WILDS macro-F1 on val',
        'threshold_candidates': len(sweep),
        'selected_threshold': selected_threshold,
        'selected_validation_macro_f1': selected_f1,
        'validation_metrics_post_nms': {
            key: float(value) for key, value in official_metrics.items()
        },
        'validation_metrics_training_rule': {
            key: float(value) for key, value in raw_metrics.items()
        },
        'test_splits_not_run': ['id_test', 'test'],
    }
    manifest_path = run_dir / 'epoch_09_inference_frozen.json'
    payload = json.dumps(manifest, indent=2, sort_keys=True) + '\n'
    if manifest_path.is_file():
        assert manifest_path.read_text(encoding='utf-8') == payload
    else:
        manifest_path.write_text(payload, encoding='utf-8')
    print('Validation images:', len(subset), '| no boxes:',
          int(np.sum(records['box_count'] == 0)))
    print(f'Selected empty threshold: {selected_threshold:.6f} '
          f'({len(sweep)} candidates; macro-F1 tuned on val)')
    print('Post-NMS image rule, held-out-camera val (threshold-tuned):')
    print(official_text.strip())
    print('Training-time max-region rule, held-out-camera val (diagnostic):')
    print(raw_text.strip())
    print('True empty fraction:', round(float(np.mean(true_y == 0)), 4),
          '| predicted empty fraction:',
          round(float(np.mean(selected_predictions == 0)), 4))
    print('Frozen inference manifest:', manifest_path)
    print('Final test splits remain untouched.')
    return manifest_path, manifest

trained_inference_manifest_path, trained_inference_manifest = (
    evaluate_and_freeze_validation()
)

**One-time final test evaluation.** The epoch-9 checkpoint, post-NMS image rule, and empty threshold were frozen using `id_val`/`val` before this cell. CELL 36 evaluates both official test splits with no tuning or updates, reports WILDS accuracy and macro-F1, and stores predictions/results outside Git. Rerunning it reuses the saved predictions for the same frozen configuration. Do not change the model or threshold in response to test results and then claim the same test as independent.

In [ ]:
# CELL 36
def evaluate_frozen_final_tests():
    import json

    assert trained_inference_manifest_path.is_file(), 'Run CELL 35 first'
    frozen_text = trained_inference_manifest_path.read_text(encoding='utf-8')
    frozen = json.loads(frozen_text)
    assert frozen == trained_inference_manifest
    assert frozen['run_id'] == run_id
    assert frozen['checkpoint_epoch'] == EVAL_BEST_EPOCH == 9
    assert frozen['checkpoint_sha256'] == EVAL_CHECKPOINT_SHA256
    assert hashlib.sha256(EVAL_CHECKPOINT.read_bytes()).hexdigest() == (
        EVAL_CHECKPOINT_SHA256
    )
    assert frozen['test_splits_not_run'] == ['id_test', 'test']
    assert frozen['threshold_objective'] == 'WILDS macro-F1 on val'
    assert frozen['image_size'] == TRAIN_RECIPE['image_size']
    assert frozen['nms_floor'] == EVAL_NMS_FLOOR
    assert frozen['nms_iou'] == EVAL_NMS_IOU
    assert frozen['nms_max_boxes'] == EVAL_NMS_MAX_BOXES
    assert frozen['animal_class_ids'] == list(animal_class_ids)
    assert frozen['animal_prompts'] == list(animal_prompts)
    assert frozen['ultralytics_version'] == ultralytics.__version__
    assert torch.equal(
        baseline_vectors.detach().cpu(), eval_checkpoint_state['class_vectors']
    ), 'Reload the frozen epoch-9 vectors with CELL 34'
    threshold = float(frozen['selected_threshold'])
    assert 0 <= threshold <= 1
    frozen_sha256 = hashlib.sha256(frozen_text.encode('utf-8')).hexdigest()
    integrated_model.eval()
    assert all(not p.requires_grad for p in integrated_model.parameters())
    results = {}
    for split_name in ('id_test', 'test'):
        subset = dataset.get_subset(split_name)
        config = {
            'protocol': 'trained-yoloclip-final-test-v1',
            'split': split_name, 'images': len(subset), 'run_id': run_id,
            'frozen_manifest_sha256': frozen_sha256,
            'checkpoint_sha256': EVAL_CHECKPOINT_SHA256,
            'threshold': threshold, 'image_size': TRAIN_RECIPE['image_size'],
            'nms_floor': EVAL_NMS_FLOOR, 'nms_iou': EVAL_NMS_IOU,
            'nms_max_boxes': EVAL_NMS_MAX_BOXES,
        }
        config_json = json.dumps(config, sort_keys=True)
        signature = hashlib.sha256(config_json.encode('utf-8')).hexdigest()[:16]
        cache_path = run_dir / f'final_{split_name}_{signature}.npz'
        if cache_path.is_file():
            with np.load(cache_path, allow_pickle=False) as cache:
                assert str(cache['config_json'].item()) == config_json
                true_y = cache['true_y'].copy()
                predicted_y = cache['predicted_y'].copy()
                box_count = cache['box_count'].copy()
            print(f'Reused frozen {split_name} predictions:', cache_path)
        else:
            loader = DataLoader(
                WILDSImageLabels(subset, TRAIN_RECIPE['image_size']),
                batch_size=TRAIN_RECIPE['batch_size'], shuffle=False,
                num_workers=0, pin_memory=True,
            )
            truth, predictions, counts = [], [], []
            with torch.inference_mode():
                for images_cpu, labels_cpu, _ in tqdm(
                    loader, desc=f'Final {split_name} inference', mininterval=10
                ):
                    output = integrated_model.predict(
                        images_cpu.to(baseline_vectors.device),
                        txt_feats=baseline_vectors,
                    )
                    boxes_per_image = trained_nms.non_max_suppression(
                        output[0], conf_thres=EVAL_NMS_FLOOR,
                        iou_thres=EVAL_NMS_IOU, max_det=EVAL_NMS_MAX_BOXES,
                        nc=len(animal_class_ids),
                    )
                    assert len(boxes_per_image) == len(labels_cpu)
                    truth.extend(labels_cpu.tolist())
                    for boxes in boxes_per_image:
                        counts.append(len(boxes))
                        if len(boxes):
                            top = boxes[boxes[:, 4].argmax()]
                            score = float(top[4].item())
                            predicted = int(top[5].item()) + 1 if (
                                score >= threshold
                            ) else 0
                        else:
                            predicted = 0
                        predictions.append(predicted)
            true_y = np.asarray(truth, dtype=np.int64)
            predicted_y = np.asarray(predictions, dtype=np.int64)
            box_count = np.asarray(counts, dtype=np.uint8)
            assert len(true_y) == len(predicted_y) == len(box_count) == len(subset)
            assert np.array_equal(true_y, subset.y_array.numpy())
            temporary_path = cache_path.with_suffix('.tmp')
            with temporary_path.open('wb') as stream:
                np.savez_compressed(
                    stream, config_json=np.asarray(config_json),
                    true_y=true_y, predicted_y=predicted_y,
                    box_count=box_count,
                )
            os.replace(temporary_path, cache_path)
            print(f'Saved frozen {split_name} predictions:', cache_path)
        assert len(true_y) == len(predicted_y) == len(box_count) == len(subset)
        assert np.array_equal(true_y, subset.y_array.numpy())
        assert np.all((predicted_y >= 0) & (predicted_y < dataset.n_classes))
        official, official_text = dataset.eval(
            torch.from_numpy(predicted_y.copy()),
            subset.y_array, subset.metadata_array,
        )
        results[split_name] = {
            'images': len(subset),
            'official_metrics': {key: float(value) for key, value in official.items()},
            'true_empty_fraction': float(np.mean(true_y == 0)),
            'predicted_empty_fraction': float(np.mean(predicted_y == 0)),
            'no_box_fraction': float(np.mean(box_count == 0)),
            'box_cap_fraction': float(np.mean(box_count == EVAL_NMS_MAX_BOXES)),
            'prediction_cache': str(cache_path),
        }
        print(f'{split_name}: {len(subset):,} images; frozen threshold={threshold:.6f}')
        print(official_text.strip())
        print('True empty:', f"{results[split_name]['true_empty_fraction']:.1%}",
              '| predicted empty:',
              f"{results[split_name]['predicted_empty_fraction']:.1%}",
              '| no boxes:', f"{results[split_name]['no_box_fraction']:.1%}")
    summary = {
        'kind': 'trained_yoloclip_frozen_final_test_v1',
        'frozen_manifest_sha256': frozen_sha256,
        'checkpoint_epoch': EVAL_BEST_EPOCH,
        'checkpoint_sha256': EVAL_CHECKPOINT_SHA256,
        'threshold': threshold, 'splits': results,
    }
    payload = json.dumps(summary, indent=2, sort_keys=True) + '\n'
    summary_path = run_dir / f'final_test_epoch_09_{frozen_sha256[:12]}.json'
    if summary_path.is_file():
        assert summary_path.read_text(encoding='utf-8') == payload
    else:
        summary_path.write_text(payload, encoding='utf-8')
    print('Final test summary:', summary_path)
    print('Model, threshold, and image rule were fixed before test inference.')
    return summary_path, summary

final_test_summary_path, final_test_summary = evaluate_frozen_final_tests()

**Preserve the selected baseline.** CELL 37 uploads the frozen epoch-9 checkpoint, the validation-selected inference manifest, and the final test summary together as a small W&B model artifact with a `final` alias. Earlier per-epoch checkpoints were already uploaded; this bundle makes the chosen version and its evaluation settings explicit. No notebook, dataset images, prediction caches, or API key are uploaded.

In [ ]:
# CELL 37
def upload_final_yoloclip_bundle():
    import json

    assert EVAL_BEST_EPOCH == 9
    assert EVAL_CHECKPOINT.is_file()
    assert trained_inference_manifest_path.is_file()
    assert final_test_summary_path.is_file(), 'Complete CELL 36 first'
    assert baseline_wandb_run.id == run_id and wandb.run is baseline_wandb_run
    assert baseline_wandb_run.settings.disable_code
    assert hashlib.sha256(EVAL_CHECKPOINT.read_bytes()).hexdigest() == (
        EVAL_CHECKPOINT_SHA256
    )
    frozen = json.loads(trained_inference_manifest_path.read_text(encoding='utf-8'))
    final = json.loads(final_test_summary_path.read_text(encoding='utf-8'))
    assert frozen == trained_inference_manifest
    assert final == final_test_summary
    assert frozen['checkpoint_epoch'] == final['checkpoint_epoch'] == 9
    assert frozen['checkpoint_sha256'] == final['checkpoint_sha256'] == (
        EVAL_CHECKPOINT_SHA256
    )
    assert set(final['splits']) == {'id_test', 'test'}
    files = {
        'epoch_09.pt': EVAL_CHECKPOINT,
        'frozen_inference.json': trained_inference_manifest_path,
        'final_test_summary.json': final_test_summary_path,
    }
    file_hashes = {
        name: hashlib.sha256(path.read_bytes()).hexdigest()
        for name, path in files.items()
    }
    marker_path = run_dir / 'final_epoch_09_wandb_upload.json'
    if marker_path.is_file():
        marker = json.loads(marker_path.read_text(encoding='utf-8'))
        assert marker['run_id'] == run_id and marker['file_sha256'] == file_hashes
        print('Final bundle already uploaded:', marker['artifact_name'])
        print('W&B run:', baseline_wandb_run.url)
        return marker
    artifact = wandb.Artifact(
        f'yoloclip-final-{run_id}', type='model',
        metadata={
            'run_id': run_id, 'selected_epoch': EVAL_BEST_EPOCH,
            'checkpoint_sha256': EVAL_CHECKPOINT_SHA256,
            'empty_threshold': float(frozen['selected_threshold']),
            'image_rule': 'highest post-NMS detection or empty below threshold',
        },
    )
    for name, path in files.items():
        artifact.add_file(str(path), name=name)
    uploaded = baseline_wandb_run.log_artifact(
        artifact, aliases=['final', 'best-id-val', 'epoch-9']
    ).wait()
    marker = {
        'run_id': run_id, 'artifact_name': uploaded.name,
        'collection': f'yoloclip-final-{run_id}',
        'file_sha256': file_hashes,
    }
    marker_path.write_text(
        json.dumps(marker, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )
    print('Final W&B model artifact:', uploaded.name)
    print('W&B run:', baseline_wandb_run.url)
    print('Uploaded files:', list(files))
    print('Epoch-9 checkpoint SHA-256:', EVAL_CHECKPOINT_SHA256)
    return marker

final_wandb_bundle = upload_final_yoloclip_bundle()